# Pelatihan Model YOLO11n untuk VNetra (Navigasi Tunanetra)
Notebook ini dibuat untuk berjalan di **Kaggle**.
Pastikan Anda telah mengaktifkan GPU dengan cara masuk ke menu: `Runtime > Change runtime type > Hardware accelerator > T4 GPU`.

Notebook ini akan mengeksekusi *pipeline* berikut:
1. Mengunduh dataset kustom dari Roboflow (Pothole, Tactile Paving, Drain, dll).
2. Mengunduh Subset COCO untuk mencegah *Catastrophic Forgetting*.
3. Menyatukan dataset dengan ID kelas (Class ID) yang selaras (25 Classes).
4. Melatih model YOLO11n menggunakan `ultralytics` dengan *augmentation* khusus kamera OV2640.
5. Mengekspor model menjadi `.tflite` (FP16 & INT8).

In [ ]:
!pip install ultralytics roboflow pyyaml fiftyone
import os
import shutil
import yaml
import glob
from roboflow import Roboflow
import fiftyone as fo
import fiftyone.zoo as foz

print("Environment siap!")

## 1. Unduh Dataset Kustom dari Roboflow
Menarik semua dataset rintangan spesifik tunanetra dari Roboflow Universe.

In [ ]:
# Menggunakan API Key Roboflow (Isi dengan API Key Anda sendiri sebelum menjalankan)
rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY_HERE")

# 1. Pothole Dataset
dataset_pothole = rf.workspace("yeeun-kim-fyvoj").project("pothole-vhmow").version(18).download("yolov11")

# 2. Tactile Paving Dataset
dataset_tactile = rf.workspace("raihan-aria").project("paving-tactile-detection").version(4).download("yolov11")

# 3. Open Drain Dataset
dataset_drain = rf.workspace("chaitanya-kharche").project("drain-overflow").version(2).download("yolov11")

# 4. Puddle Dataset
dataset_puddle = rf.workspace("ambitious-jda7x").project("puddle-zlrsu").version(2).download("yolov11")

# 5. Speed Bump Dataset
dataset_speedbump = rf.workspace("road-safety").project("speed-bump-tonyt").version(3).download("yolov11")

# 6. Pole Dataset
dataset_pole = rf.workspace("ghost-gsj7h").project("utility-pole-aka9k").version(3).download("yolov11")

# 7. Hanging Branch Dataset
dataset_branch = rf.workspace("utem").project("branch-7qne7").version(2).download("yolov11")

# 8. Stairs Dataset
dataset_stairs = rf.workspace("jatin-sne2e").project("stairs-zqsvn").version(2).download("yolov11")

# 9. Curb Ramp Dataset (Undakan Trotoar)
dataset_curb = rf.workspace("sidewalkdetect").project("-sidewalk").version(10).download("yolov11")



## 2. Unduh Dataset Bawaan COCO (Anti-Catastrophic Forgetting)
Mencegah YOLO melupakan wujud manusia, mobil, atau motor karena tertimpa data Roboflow di atas. Kita mengambil 3000 gambar subset dari COCO-2017 menggunakan `fiftyone`.

In [ ]:
coco_classes = [
    "person", "bicycle", "car", "motorcycle", "bus", "truck", "train", 
    "fire hydrant", "stop sign", "parking meter", "bench", "chair", "potted plant", "dog", "cat"
]

print("Mulai mengunduh subset COCO-2017 (Estimasi waktu: 3-5 menit)...")
coco_dataset = foz.load_zoo_dataset(
    "coco-2017",
    split="train",
    label_types=["detections"],
    classes=coco_classes,
    max_samples=3000, # Mengambil 3000 gambar
)

coco_export_dir = "/kaggle/working/coco_subset"
print(f"Mengekspor COCO ke format YOLO di {coco_export_dir}...")
coco_dataset.export(
    export_dir=coco_export_dir,
    dataset_type=fo.types.YOLOv5Dataset,
    classes=coco_classes
)

# Menyelaraskan struktur folder 'val' ke 'valid' jika fiftyone menggunakan 'val'
if os.path.exists(f"{coco_export_dir}/val"):
    os.rename(f"{coco_export_dir}/val", f"{coco_export_dir}/valid")


## 3. Penggabungan (Merging) Seluruh Dataset
Menyatukan seluruh dataset (10 Roboflow + 1 COCO) ke dalam folder `vnetra_master_dataset` sambil merekayasa ID Kelas mereka agar berurutan (0-24) secara konsisten.

In [ ]:
master_dir = "/kaggle/working/vnetra_master_dataset"

for split in ['train', 'valid', 'test']:
    os.makedirs(f"{master_dir}/{split}/images", exist_ok=True)
    os.makedirs(f"{master_dir}/{split}/labels", exist_ok=True)

# Definisi urutan Master Class VNetra (COCO + Custom)
master_classes = [
    'person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck', 'train', 
    'fire hydrant', 'stop sign', 'parking meter', 'bench', 'chair', 'potted plant', 'dog', 'cat',
    'pothole', 'open_drain', 'puddle', 'speed_bump', 'pole', 
    'hanging_branch', 'tactile_paving', 'stairs_up', 'stairs_down', 'curb'
]
master_class_to_id = {name: i for i, name in enumerate(master_classes)}

def merge_dataset(source_path, original_classes, class_mapping):
    # Jika path tidak ada, abaikan (terutama jika split test tidak ada)
    if not os.path.exists(source_path): return
    
    for split in ['train', 'valid', 'test']:
        img_dir = f"{source_path}/{split}/images"
        if not os.path.exists(img_dir):
            continue
        
        for img_path in glob.glob(f"{img_dir}/*"):
            file_name = os.path.basename(img_path)
            
            # Cari file txt label yang bersesuaian
            lbl_name = file_name.rsplit('.', 1)[0] + '.txt'
            lbl_path = f"{source_path}/{split}/labels/{lbl_name}"
            
            if not os.path.exists(lbl_path):
                continue
                
            # Baca dan konversi class_id pada label
            new_labels = []
            with open(lbl_path, 'r') as f:
                lines = f.readlines()
            
            valid_objects = 0
            for line in lines:
                parts = line.strip().split()
                if len(parts) < 5: continue
                
                orig_id = int(parts[0])
                # Cegah error jika dataset memiliki ID cacat
                if orig_id >= len(original_classes): continue
                
                class_name = original_classes[orig_id]
                if class_name in class_mapping:
                    new_id = master_class_to_id[class_mapping[class_name]]
                    new_labels.append(f"{new_id} {' '.join(parts[1:])}\n")
                    valid_objects += 1
            
            # Jika gambar mengandung objek yang valid bagi VNetra, copy image dan tulis label
            if valid_objects > 0:
                # Tambahkan awalan dataset_name agar tidak terjadi overwrite gambar yang sama
                prefix = source_path.split('/')[-1]
                new_img_name = f"{prefix}_{file_name}"
                new_lbl_name = f"{prefix}_{lbl_name}"
                
                shutil.copy(img_path, f"{master_dir}/{split}/images/{new_img_name}")
                with open(f"{master_dir}/{split}/labels/{new_lbl_name}", 'w') as f:
                    f.writelines(new_labels)

print("Memproses COCO Subset (Mencegah Catastrophic Forgetting)...")
merge_dataset(coco_export_dir, coco_classes, {c: c for c in coco_classes})

print("Memproses Pothole Dataset...")
merge_dataset(dataset_pothole.location, ["pothole"], {"pothole": "pothole"})

print("Memproses Tactile Paving Dataset...")
merge_dataset(dataset_tactile.location, ["stop", "0", "2", "3", "4", "Go"], {"stop": "tactile_paving", "Go": "tactile_paving"})

print("Memproses Open Drain Dataset...")
merge_dataset(dataset_drain.location, ["closed manhole(not overflowing)", "drainage overflow(repair)", "fake manhole", "open drainage(not overflowing)", "open drainage(overflowing)", "open manhole(not overflowing)"], {"open drainage(not overflowing)": "open_drain", "open drainage(overflowing)": "open_drain", "drainage overflow(repair)": "open_drain"})

print("Memproses Puddle Dataset...")
merge_dataset(dataset_puddle.location, ["puddle"], {"puddle": "puddle"})

print("Memproses Speed Bump Dataset...")
merge_dataset(dataset_speedbump.location, ["speed bump", "speed bump "], {"speed bump": "speed_bump", "speed bump ": "speed_bump"})

print("Memproses Pole Dataset...")
merge_dataset(dataset_pole.location, ["defect", "insulator", "pole", "faulty_insulator", "pole_including_insulator"], {"pole": "pole"})

print("Memproses Hanging Branch Dataset...")
merge_dataset(dataset_branch.location, ["0", "Branch", "branches"], {"Branch": "hanging_branch", "branches": "hanging_branch"})

print("Memproses Stairs Dataset...")
merge_dataset(dataset_stairs.location, ["downstair", "objects", "upstair"], {"downstair": "stairs_down", "upstair": "stairs_up"})

print("Memproses Curb Ramp Dataset...")
merge_dataset(dataset_curb.location, ["blind tracks", "curb ramp", "zebra crossing"], {"curb ramp": "curb"})


# Buat file master data.yaml untuk pelatihan
yaml_content = {
    "path": master_dir,
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": len(master_classes),
    "names": master_classes
}

with open(f"{master_dir}/data.yaml", 'w') as f:
    yaml.dump(yaml_content, f, sort_keys=False)

print(f"Master Dataset berhasil dirakit! Path YAML: {master_dir}/data.yaml")

## 4. Training YOLO11n dengan Augmentasi OV2640
Melatih model YOLOv11 versi Nano dengan augmentasi hyperparameter yang dikhususkan untuk **motion blur, rotasi sudut jalan, dan fluktuasi pencahayaan** yang sering ditemui pada kamera OV2640 di perangkat wearable.

In [ ]:
from ultralytics import YOLO

# Load pre-trained model (YOLO11 Nano)
model = YOLO('yolo11n.pt')

# Mulai proses training
results = model.train(
    data=f"{master_dir}/data.yaml",
    epochs=50,             # Tambahkan epochs (misal 100) jika butuh hasil lebih akurat
    imgsz=640,             # Sesuai dengan resolusi VGA kamera ESP32-S3 (OV2640)
    batch=32,
    device=0,              # Memastikan pemakaian hardware GPU Colab (T4)
    project='vnetra_training',
    name='yolo11n_custom',
    
    # --- HYPERPARAMETER AUGMENTASI UNTUK KAMERA OV2640 ---
    mosaic=1.0,      # Menggabungkan 4 gambar jadi 1 untuk ketangguhan deteksi terpotong
    degrees=15.0,    # Toleransi rotasi untuk simulasi goyangan kamera saat berjalan
    hsv_h=0.015,     # Simulasi fluktuasi cahaya matahari/lampu jalan
    hsv_s=0.7,       # Simulasi warna pudar kamera analog
    hsv_v=0.4,       # Simulasi kontras rendah / kondisi backlight matahari
)

## 5. Export ke TensorFlow Lite (TFLite)
Mengekspor bobot model menjadi format `.tflite` dalam dua bentuk kuantisasi:
1. **FP16** (Half Precision) -> Sangat efisien dan kompatibel untuk *GPU Delegation* di Android.
2. **INT8** (Full Integer) -> Wajib untuk akselerator *NPU / NNAPI* yang membutuhkan model super ringan.

In [ ]:
print("Mengekspor model...")
# 1. Export ke TFLite (FP16) - Optimal untuk GPU Mobile
export_fp16 = model.export(format="tflite", half=True, optimize=True)

# 2. Export ke TFLite (INT8) - Wajib untuk NPU / NNAPI
# Note: INT8 butuh representative dataset untuk kalibrasi yang akurat
export_int8 = model.export(format="tflite", int8=True, data=f"{master_dir}/data.yaml", optimize=True)

print("===========================================================")
print("Model berhasil diekspor! Lokasi file TFLite:")
print("FP16:", export_fp16)
print("INT8:", export_int8)
print("===========================================================")

# Cara Mengunduh: 
# Buka menu folder di bilah kiri Colab, navigasikan ke path di atas
# (biasanya di folder: vnetra_training/yolo11n_custom/weights/), klik kanan -> Download pada file tflite.

## 6. Validasi Kuantisasi (Benchmarking Skripsi)
Menguji kembali model pada Validation Set untuk melihat seberapa jauh penurunan akurasi (mAP) akibat proses kompresi FP16 dan INT8 dibanding model aslinya.

In [ ]:
!pip install tensorflow
import gc
gc.collect()

print("\n=== EVALUASI MODEL ASLI (.pt) ===")
val_pt = model.val(data=f"{master_dir}/data.yaml")
map_pt = val_pt.box.map50

print("\n=== EVALUASI MODEL FP16 (.tflite) ===")
model_fp16 = YOLO(export_fp16, task='detect')
val_fp16 = model_fp16.val(data=f"{master_dir}/data.yaml")
map_fp16 = val_fp16.box.map50

print("\n=== EVALUASI MODEL INT8 (.tflite) ===")
model_int8 = YOLO(export_int8, task='detect')
val_int8 = model_int8.val(data=f"{master_dir}/data.yaml")
map_int8 = val_int8.box.map50

print("\n=========================================")
print("KESIMPULAN PERBANDINGAN mAP@50 (Akurasi):")
print(f"Original (.pt)   : {map_pt:.4f}")
print(f"FP16 (.tflite)   : {map_fp16:.4f}")
print(f"INT8 (.tflite)   : {map_int8:.4f}")
print("=========================================")


## 7. Pengujian Visualisasi Langsung (Predict)
Mengambil satu gambar tes secara acak dan menampilkan prediksi kotak deteksi dari model asli (.pt) vs model terkompresi (.tflite) agar Anda bisa meletakkannya di Laporan Skripsi.

In [ ]:
import random
import matplotlib.pyplot as plt
import cv2

# Pilih satu gambar acak dari dataset test (atau valid jika test tidak ada)
test_images = glob.glob(f"{master_dir}/valid/images/*.jpg")
if test_images:
    test_img = random.choice(test_images)
    print(f"Menguji gambar: {test_img}")
    
    # Prediksi pakai model Asli
    res_pt = model.predict(source=test_img, imgsz=640)
    img_pt = res_pt[0].plot()
    
    # Prediksi pakai model INT8
    res_int8 = model_int8.predict(source=test_img, imgsz=640)
    img_int8 = res_int8[0].plot()
    
    # Tampilkan perbandingan
    fig, ax = plt.subplots(1, 2, figsize=(15, 7))
    ax[0].imshow(cv2.cvtColor(img_pt, cv2.COLOR_BGR2RGB))
    ax[0].set_title("Prediksi Model Asli (.pt)")
    ax[0].axis("off")
    
    ax[1].imshow(cv2.cvtColor(img_int8, cv2.COLOR_BGR2RGB))
    ax[1].set_title("Prediksi Model INT8 (.tflite)")
    ax[1].axis("off")
    
    plt.tight_layout()
    plt.show()
else:
    print("Tidak ada gambar di folder valid untuk diprediksi.")
